In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import altair as alt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

# LOAD DATA

In [11]:
meta=pd.read_excel("/Users/jacobg/Dropbox/shared_leif/Jacob/metadata/wes_meta_JG202504.xlsx")
meta_wes = meta.dropna(subset=['wes tumor pre', 'wes tumor sg', 'wes extra FFT'], how='all')
meta_paired = meta_wes.dropna(subset='wes normal').set_index('new name', drop=False)
bad_samples = ['BIDMC2_pre', 'BIDMC3_post', 'BIDMC4_pre', 'BIDMC4_post', 'BIDMC5_post', 'BIDMC6_pre', 
               'BIDMC6_post', 'BIDMC7_post', 'DFC7_pre_FFT', 'DFC9_pre', 'DFC12_post', 'DFC19_pre', 
               'DFC19_pre', 'DFC20_post', 'DFC23_post', 'MGH3_post', 'MGH5_post', 'MGNS4_pre', 
               'NWH2_pre_FFT', 'MGNS5_post', 'DFC1_pre', 'DFC19_post', 'MGNS4_post', 'BIDMC3_post', 
               'DFC1_post']
for c in ['wes tumor pre', 'wes tumor sg', 'wes extra FFT', 'wes normal']:
    meta_paired[c] = meta_paired[c].replace(bad_samples, np.nan)



tumor_meta = pd.DataFrame({'tumor': pd.concat([meta_paired['wes tumor pre'], meta_paired['wes tumor sg'], meta_paired['wes extra FFT']])}).dropna()
tumor_meta = tumor_meta.reset_index().rename(columns={'new name': 'patient'}).set_index('tumor', drop=False)
tumor_meta['patient'] = ''
tumor_meta['timing'] = ''
tumor_meta['sg_response'] = ''
tumor_meta['final_response'] = ''
tumor_meta['plotname'] = ''

for patient in meta_paired.index:
    for group in ['wes tumor pre', 'wes tumor sg', 'wes extra FFT']:
        tumor = meta_paired.at[patient, group]
        if pd.isna(tumor):
            continue
        tumor_meta.at[tumor,'patient'] = patient
        tumor_meta.at[tumor,'sg_response'] = meta_paired.at[patient,'post sg response']
        tumor_meta.at[tumor,'final_response'] = meta_paired.at[patient,'final response(surgery)']
        if group in ['wes tumor pre', 'wes extra FFT']:
            tumor_meta.at[tumor,'plotname'] = meta_paired.at[patient,'wes pre plotname']
        else:
            tumor_meta.at[tumor,'plotname'] = meta_paired.at[patient,'wes post plotname']
        
tumor_meta['final_response_binary'] = tumor_meta['final_response'].apply(lambda x: 'pCR' if x == 'pCR' else 'RD')
tumor_meta['timing'] = tumor_meta['tumor'].apply(lambda x: 'post' if x.endswith("post") else 'pre')
tumor_meta['tissue_type'] = tumor_meta['tumor'].apply(lambda x: 'FFT' if x.endswith("FFT") else 'frozen')
tumor_meta.sort_values(by='patient', axis=0, inplace=True)

display(tumor_meta)
print(tumor_meta.shape)


,patient,tumor,timing,sg_response,final_response,plotname,final_response_binary,tissue_type
tumor,,,,,,,,
BIDMC1_pre,BIDMC1,BIDMC1_pre,pre,RD,RCB-I,P34R,RD,frozen
BIDMC3_pre,BIDMC3,BIDMC3_pre,pre,pCR,pCR,P38R,pCR,frozen
BIDMC5_pre,BIDMC5,BIDMC5_pre,pre,RD,RCB-II,P42R,RD,frozen
BIDMC7_pre,BIDMC7,BIDMC7_pre,pre,RD,RCB-I,P47R,RD,frozen
BIDMC8_pre,BIDMC8,BIDMC8_pre,pre,pCR,pCR,P49R,pCR,frozen
DFC1_pre_FFT,DFC1,DFC1_pre_FFT,pre,RD,RCB-I,P02R,RD,FFT
DFC10_pre,DFC10,DFC10_pre,pre,RD,pCR,P20R,pCR,frozen
DFC11_pre,DFC11,DFC11_pre,pre,pCR,pCR,P22R,pCR,frozen
DFC13_post,DFC13,DFC13_post,post,RD,RCB-II,P26T,RD,frozen


(45, 8)


# LOAD SIGNATURES

In [12]:
path = '/Users/jacobg/Dropbox/shared_leif/Jacob/WES_2024/signatures/SigProfiler_pass_mutants/Assignment_Solution/Activities/Assignment_Solution_Activities.txt'
sbs_sigs = pd.read_csv(path, sep="\t")
sbs_sigs = sbs_sigs[sbs_sigs['Samples'].isin(bad_samples) == False]
sbs_sigs['patient'] = sbs_sigs.apply(lambda row: tumor_meta.at[row['Samples'], 'patient'], axis=1)
sbs_sigs['response'] = sbs_sigs.apply(lambda row: tumor_meta.at[row['Samples'], 'sg_response'], axis=1)
sbs_sigs['timing'] = sbs_sigs.apply(lambda row: tumor_meta.at[row['Samples'], 'timing'], axis=1)
sbs_sigs['sample_group'] = sbs_sigs.apply(lambda row: row['timing'] + '-' + row['response'], axis=1)
sbs_sigs['plotname'] = sbs_sigs.apply(lambda row: tumor_meta.at[row['Samples'], 'plotname'], axis=1)
sbs_sigs

,Samples,SBS1,SBS2,SBS3,SBS4,SBS5,SBS6,SBS7a,SBS7b,SBS7c,...,SBS94,SBS96,SBS97,SBS98,SBS99,patient,response,timing,sample_group,plotname
0,BIDMC1_pre,80,0,248,0,79,86,0,0,0,...,0,0,0,0,0,BIDMC1,RD,pre,pre-RD,P34R
3,BIDMC3_pre,127,0,0,0,137,116,0,0,0,...,0,0,0,0,0,BIDMC3,pCR,pre,pre-pCR,P38R
7,BIDMC5_pre,60,0,0,0,188,0,0,0,0,...,0,181,0,0,0,BIDMC5,RD,pre,pre-RD,P42R
11,BIDMC7_pre,39,34,0,0,52,0,0,0,0,...,0,56,0,0,0,BIDMC7,RD,pre,pre-RD,P47R
12,BIDMC8_pre,71,0,0,0,408,149,0,0,0,...,0,0,0,0,0,BIDMC8,pCR,pre,pre-pCR,P49R
13,DFC10_pre,49,45,0,0,146,0,0,0,0,...,0,0,0,0,0,DFC10,RD,pre,pre-RD,P20R
14,DFC11_pre,74,0,0,0,246,0,0,0,0,...,0,0,0,0,0,DFC11,pCR,pre,pre-pCR,P22R
16,DFC13_post,56,0,0,0,45,0,0,0,0,...,0,0,0,0,0,DFC13,RD,post,post-RD,P26T
17,DFC13_pre,274,0,0,0,257,0,0,0,0,...,0,0,0,0,0,DFC13,RD,pre,pre-RD,P26R
18,DFC14_pre,35,0,0,89,137,0,0,0,0,...,0,0,0,0,0,DFC14,pCR,pre,pre-pCR,P27R


In [13]:
# sbs_sigs = tumor_meta.reset_index(drop=True).merge(sbs_sigs, how='inner', left_on='tumor', right_on='Samples', validate='one_to_one')
# sbs_sigs

In [15]:
# Identify SBS signature columns from the dataframe present in at least 1 sample
signature_order = [col for col in sbs_sigs.columns if col.startswith("SBS") and sbs_sigs[col].sum() != 0]
signature_order.sort(key=lambda x: (len(x), x))

# 2. Define the Mapping based on your table
sbs_mapping_raw = {
    "MMR deficiency": ["SBS6", "SBS14", "SBS15", "SBS20", "SBS21", "SBS26", "SBS44"],
    "POL deficiency": ["SBS10a", "SBS10b", "SBS10c", "SBS10d", "SBS28"],
    "HR deficiency": ["SBS3"],
    "BER deficiency": ["SBS30", "SBS36"],
    "Treatment/Chemo": ["SBS11", "SBS25", "SBS31", "SBS32", "SBS35", "SBS86", "SBS87", "SBS90", "SBS99"],
    "APOBEC": ["SBS2", "SBS13"],
    "Tobacco": ["SBS4", "SBS29", "SBS92", "SBS100", "SBS109"],
    "UV": ["SBS7a", "SBS7b", "SBS7c", "SBS7d", "SBS38"],
    "AA (Aristolochic acid)": ["SBS22a", "SBS22b"],
    "Colibactin": ["SBS88"],
    "Artifact": ["SBS27", "SBS43", "SBS45", "SBS46", "SBS47", "SBS48", "SBS49", "SBS50", 
                 "SBS51", "SBS52", "SBS53", "SBS54", "SBS55", "SBS56", "SBS57", "SBS58", 
                 "SBS59", "SBS60", "SBS95"],
    "Lymphoid": ["SBS9", "SBS84", "SBS85"],
    "Clock-like/Aging": ["SBS1", "SBS5"], # Common signatures not in your table
    "Unknown/Other": ["SBS8", "SBS12", "SBS16", "SBS17a", "SBS17b", "SBS18", "SBS19", 
                      "SBS23", "SBS24", "SBS33", "SBS34", "SBS37", "SBS39", "SBS40a", 
                      "SBS40b", "SBS40c", "SBS41", "SBS42", "SBS89", "SBS91", "SBS93", 
                      "SBS94", "SBS96", "SBS97", "SBS98"]
}

# Flatten the mapping for easy lookup
sbs_to_group = {}
for group, sigs in sbs_mapping_raw.items():
    for sig in sigs:
        sbs_to_group[sig] = group

# Create Descriptions (Using Group + ID)
sbs_descriptions = {}
for sig in signature_order:
    group = sbs_to_group.get(sig, "Other")
    sbs_descriptions[sig] = f"{sig} ({group})"

# Create the name order for sorting the Y-axis
signature_name_order = [sbs_descriptions.get(sig, sig) for sig in signature_order]

# Normalize the data
sbs_sigs_norm = sbs_sigs.copy()
signature_sums = sbs_sigs[signature_order].sum(axis=1)
#signature_sums = signature_sums.replace(0, 1) 
for col in signature_order:
    sbs_sigs_norm[col] = sbs_sigs[col] / signature_sums

# Melt the dataframe
sbs_sigs_long = sbs_sigs_norm.melt(
    id_vars=['patient','plotname', 'timing', 'response', 'sample_group'], # adjust as needed
    value_vars=signature_order,
    var_name='Signature',
    value_name='Value'
)

# Map signatures to their descriptions and groups
sbs_sigs_long['Signature Name'] = sbs_sigs_long['Signature'].map(sbs_descriptions)
sbs_sigs_long['Signature Group'] = sbs_sigs_long['Signature'].map(sbs_to_group).fillna("Other")

# Create the Chart
y_axis = alt.Axis(labelLimit=0, labelOverlap=False)
chart = alt.Chart(sbs_sigs_long).mark_circle().encode(
    x=alt.X('plotname:N', title='Sample'),
    y=alt.Y('Signature Name:N', sort=signature_name_order, axis=y_axis),
    # size=alt.Size('Value:Q', title='Contribution', scale=alt.Scale(range=[0, 400])),
    size=alt.Size('Value:Q', title='Contribution'),
    color=alt.Color('Signature Group:N', title='Etiology Group'),
    tooltip=['patient', 'Signature', 'Value']
)

# Configure the Chart
chart = chart.facet(
    column=alt.Column('sample_group:N', title='Sample Group', sort=['pre-pCR','pre-RD','post-RD'])
).resolve_scale(
    x='independent'
).properties(
    title='Normalized SBS Signatures by Response'
).configure_view(
    stroke=None
).configure_axis(
    grid=False
).configure_legend(
    orient='right',
    labelFontSize=12,
    titleFontSize=14
)

# Display
display(chart)

alt.FacetChart(...)